# Step 3. Adaptation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/younesStrittmatter/sweetpea-starter/blob/main/tutorial/notebooks/3_adaptation.ipynb)

The tutorial step without an assistant. Run the cells from top to bottom; the tables below are from one run, yours will differ where the design leaves things open.

In [ ]:
%pip install -q sweetpea pandas

**So far.** Step 1 crossed the ink colour with congruency: half of the trials congruent. This step changes that share, and the share becomes the manipulation.

**The question.** Do people adjust to how frequent conflict is?

**Prediction.** The congruency effect is smaller in mostly incongruent blocks than in mostly congruent blocks.

**The alternative to rule out.** In a mostly congruent block the word predicts the ink, so people can simply learn to follow the word.

**The analysis.** The congruency effect in mostly congruent blocks against the congruency effect in mostly incongruent blocks.

### As in step 1

In [ ]:
import pandas as pd
from sweetpea import (Factor, DerivedLevel, WithinTrial, Transition, MinimumTrials,
                      CrossBlock, synthesize_trials, tabulate_experiments, CMSGen)

colours = ["red", "green", "blue"]

# regular factors: set freely on every trial
colour = Factor("colour", colours)      # the ink
word = Factor("word", colours)          # what the word says

In [ ]:
def same(colour, word):
    return colour == word


def different(colour, word):
    return colour != word

### New: a weight

A level can have a weight: the third argument of `DerivedLevel`. 3 to 1 makes the first level three times as frequent as the second. Crossed with the ink colour that is 12 combinations, two rounds: 18 of 24 trials.

In [ ]:
mostly_congruent = Factor("congruency", [
    DerivedLevel("congruent", WithinTrial(same, [colour, word]), 3),
    DerivedLevel("incongruent", WithinTrial(different, [colour, word]), 1),
])
mostly_incongruent = Factor("congruency", [
    DerivedLevel("congruent", WithinTrial(same, [colour, word]), 1),
    DerivedLevel("incongruent", WithinTrial(different, [colour, word]), 3),
])

block_mc = CrossBlock(design=[colour, word, mostly_congruent], crossing=[colour, mostly_congruent],
                      constraints=[MinimumTrials(24)])
block_mi = CrossBlock(design=[colour, word, mostly_incongruent], crossing=[colour, mostly_incongruent],
                      constraints=[MinimumTrials(24)])

### Two blocks of each kind

Each kind of block is generated on its own. The second argument of `synthesize_trials` is the number of sequences.

In [ ]:
experiments_mc = synthesize_trials(block_mc, 2, CMSGen)
experiments_mi = synthesize_trials(block_mi, 2, CMSGen)
tabulate_experiments(block_mc, experiments_mc, [mostly_congruent])
tabulate_experiments(block_mi, experiments_mi, [mostly_incongruent])

Experiment 0:
congruency congruent   | frequency 18 | proportion 75.0%
congruency incongruent | frequency 6  | proportion 25.0%

Experiment 1:
congruency congruent   | frequency 18 | proportion 75.0%
congruency incongruent | frequency 6  | proportion 25.0%

Experiment 0:
congruency congruent   | frequency 6  | proportion 25.0%
congruency incongruent | frequency 18 | proportion 75.0%

Experiment 1:
congruency congruent   | frequency 6  | proportion 25.0%
congruency incongruent | frequency 18 | proportion 75.0%

### How well does the word tell the ink?

For each word, how often it is shown in its own colour (congruent). That number is the alternative explanation. First a mostly congruent block, then a mostly incongruent one.

In [ ]:
tabulate_experiments(block_mc, experiments_mc[:1], [word, mostly_congruent])
tabulate_experiments(block_mi, experiments_mi[:1], [word, mostly_incongruent])

Experiment 0:
word red   | congruency congruent   | frequency 6 | proportion 25.0%
word red   | congruency incongruent | frequency 1 | proportion 4.166666666666666%
word green | congruency congruent   | frequency 6 | proportion 25.0%
word green | congruency incongruent | frequency 3 | proportion 12.5%
word blue  | congruency congruent   | frequency 6 | proportion 25.0%
word blue  | congruency incongruent | frequency 2 | proportion 8.333333333333332%

Experiment 0:
word red   | congruency congruent   | frequency 2 | proportion 8.333333333333332%
word red   | congruency incongruent | frequency 5 | proportion 20.833333333333336%
word green | congruency congruent   | frequency 2 | proportion 8.333333333333332%
word green | congruency incongruent | frequency 6 | proportion 25.0%
word blue  | congruency congruent   | frequency 2 | proportion 8.333333333333332%
word blue  | congruency incongruent | frequency 7 | proportion 29.166666666666668%

### A session

Four blocks one after the other, each with its context as a column.

In [ ]:
order = [("mostly congruent", experiments_mc[0]), ("mostly incongruent", experiments_mi[0]),
         ("mostly incongruent", experiments_mi[1]), ("mostly congruent", experiments_mc[1])]
session = pd.concat([pd.DataFrame(trials).assign(block=number, context=context)
                     for number, (context, trials) in enumerate(order, start=1)], ignore_index=True)
display(session.head(4))

,colour,word,congruency,block,context
0,green,blue,incongruent,1,mostly congruent
1,blue,blue,congruent,1,mostly congruent
2,red,green,incongruent,1,mostly congruent
3,blue,blue,congruent,1,mostly congruent


**Check.** Are the proportions what was asked for? How many incongruent trials does a mostly congruent block have: is that enough to measure the effect there? A word is in its own colour far more often in one context than in the other: how could learning and control be told apart?

**Try.** Change the weights to `(2, 1)` and `(1, 2)`. What are the proportions and the block lengths now?

**Next.** Step 4 asks whether this adjustment has a price when the task changes. There the weight meets a limit: it cannot depend on another factor.